# SQL Worksheet — Week4

Use the following tables from the Riva Data Platform:

- `rivadataplatform.dataproduct.dim_batch`
- `rivadataplatform.dataproduct.dim_class`
- `rivadataplatform.dataproduct.fact_attendance`
- `rivadataplatform.dataproduct.dim_student`
- `rivadataplatform.dataproduct.dim_date`

**Instructions**
- Write SQL for each question.
- Do not modify the source data.
- Use clear aliases where JOINs are involved.
- Unless a question specifically asks for a particular column, select only the columns needed to answer it.


## Tables / Relationships

Useful keys:
- `dim_student.student_key` ↔ `fact_attendance.student_key`
- `dim_class.class_key` ↔ `fact_attendance.class_key`
- `dim_batch.batch_key` ↔ `fact_attendance.batch_key`
- `dim_class.batch_id` ↔ `dim_batch.batch_id`

## Question 1 — Complete Attendance Detail
Join all five tables and return one row per attendance record with student identity, null-safe city, batch, class date, calendar day, topic, class status, attendance status, and remarks. Replace null remarks and topics with labels and sort by date and student.

In [0]:
--Write your code here
Select 
fa.attendance_id,
ds.student_id,
ds.student_name,
coalesce(ds.city,"No City") AS city,
db.batch_name,
coalesce(dc.class_day,dd.day_name) AS day_name,
dc.class_date,
coalesce(dc.topic, 'No Topic assigned') AS topic,
dc.status AS class_status
FROM rivadataplatform.dataproduct.fact_attendance AS fa
JOIN rivadataplatform.dataproduct.dim_student AS ds
    ON ds.student_key = fa.student_key
JOIN rivadataplatform.dataproduct.dim_class AS dc
    ON dc.class_key = fa.class_key
JOIN rivadataplatform.dataproduct.dim_batch AS db
    ON db.batch_key = fa.batch_key
LEFT JOIN rivadataplatform.dataproduct.dim_date AS dd
order by dc.class_date, ds.student_name;

## Question 2 — Class Attendance Scorecard
For every class, including classes with no attendance, join class, batch, attendance, and student dimensions. Group by class and return distinct students, Present/Late/Absent counts, and total records. Label null topics and preserve zero counts.

In [0]:
--Write your code here
SELECT
count(fa.attendance_id) AS total_records,
dc.class_id,
coalesce(dc.topic, 'No topic assigned') AS topic,
count(distinct fa.student_key) AS distinct_students,
sum(case when fa.attendance_status = 'Present' then 1 else 0 end) AS present_count,
sum(case when fa.attendance_status = 'Late' then 1 else 0 end) AS late_count,
sum(case when fa.attendance_status = 'Absent' then 1 else 0 end) AS absent_count
FROM rivadataplatform.dataproduct.dim_class AS dc
LEFT JOIN rivadataplatform.dataproduct.fact_attendance AS fa
    ON fa.class_key = dc.class_key
LEFT JOIN rivadataplatform.dataproduct.dim_batch AS db
    ON db.batch_id = dc.batch_id
group by dc.class_id, coalesce(dc.topic, 'No topic assigned')
order by dc.class_id;

## Question 3 — Attendance Rate by Class Date
Join class, batch, attendance, and date dimensions. For each class date and topic, calculate total records, attended records (`Present` + `Late`), absent records, and attendance rate. Use `NULLIF`, label null topics, and return only dates with attendance.

In [0]:
--Write your code here
SELECT
count(fa.attendance_id) AS total_records,
sum(case when fa.attendance_status IN ('Present', 'Late') then 1 else 0 end) AS attended_records,
sum(case when fa.attendance_status = 'Absent' then 1 else 0 end) AS absent_records,
(sum(case when fa.attendance_status IN ('Present', 'Late') then 1 else 0 end) * 100.0) / 
nullif(count(fa.attendance_id), 0) AS attendance_rate,
coalesce(dc.topic, 'no topic assigned') AS topic,
db.batch_name,
dc.class_date

FROM rivadataplatform.dataproduct.dim_class AS dc
JOIN rivadataplatform.dataproduct.fact_attendance AS fa
    ON fa.class_key = dc.class_key
JOIN rivadataplatform.dataproduct.dim_batch AS db
    ON db.batch_key = fa.batch_key
LEFT JOIN rivadataplatform.dataproduct.dim_date AS dd
    ON dd.date_key = fa.date_key

group by dc.class_date, coalesce(dd.day_name, dc.class_day), db.batch_name, coalesce(dc.topic, 'no topic assigned')
having COUNT(fa.attendance_id) > 0
order by dc.class_date;

## Question 4 — Classes With Absence Risk
Join classes, batches, attendance, and students. Group by class, return absent count, distinct affected students, and absent percentage, and keep only classes where the absent count is greater than zero. Include null-safe topic text.

In [0]:
--Write your code here
SELECT
sum(case when fa.attendance_status = 'Absent' then 1 else 0 end) AS absent_count,
count(distinct fa.student_key) filter (where fa.attendance_status = 'Absent') AS affected_students,
(sum(case when fa.attendance_status = 'Absent' then 1 else 0 end) * 100)
        / nullif(count(fa.attendance_id), 0) AS absent_percentage,
dc.class_id

FROM rivadataplatform.dataproduct.dim_class AS dc
JOIN rivadataplatform.dataproduct.dim_batch AS db
    ON db.batch_id = dc.batch_id
JOIN rivadataplatform.dataproduct.fact_attendance AS fa
    ON fa.class_key = dc.class_key
JOIN rivadataplatform.dataproduct.dim_student AS ds
    ON ds.student_key = fa.student_key

group by dc.class_id
having sum(case when fa.attendance_status = 'Absent' then 1 else 0 end) > 0
order by absent_count desc;

## Question 5 — Student Issues by Location
Join students, attendance, classes, batches, and dates. Group by student and null-safe city, count Late and Absent records, calculate the issue rate, and return only students with at least one issue. Order by issue rate descending.

In [0]:
--Write your code here
SELECT
    ds.student_id,
    ds.student_name,
    coalesce(ds.city, 'Unknown city') AS city,
    sum(case when fa.attendance_status in ('Late', 'Absent') then 1 else 0 end ) AS issue_count,
    (sum(case when fa.attendance_status in ('Late', 'Absent')then 1 else 0 end ) * 100.0)
         / nullif(count(fa.attendance_id), 0) AS issue_rate

FROM rivadataplatform.dataproduct.dim_student AS ds
JOIN rivadataplatform.dataproduct.fact_attendance AS fa
    ON fa.student_key = ds.student_key
JOIN rivadataplatform.dataproduct.dim_class AS dc
    ON dc.class_key = fa.class_key
LEFT JOIN rivadataplatform.dataproduct.dim_batch AS db
    ON db.batch_key = fa.batch_key
LEFT JOIN rivadataplatform.dataproduct.dim_date AS dd
    ON dd.date_key = fa.date_key
group by ds.student_id, ds.student_name, coalesce(ds.city, 'Unknown city')
having sum(case when fa.attendance_status in ('Late', 'Absent') then 1 else 0 end ) > 0
order by issue_rate desc, ds.student_name;

## Question 6 — Batch Relationship Data Quality
Join attendance to class and both batch representations. Group by class and recorded/expected batch names, then return only mismatches between the attendance `batch_key` and the batch identified by `dim_class.batch_id`. Include mismatch record counts and null-safe labels.

In [0]:
--Write your code here
SELECT
    dc.class_id,
    fa.batch_key AS recorded_batch_key,
    dc.batch_id AS expected_batch_key,
    coalesce(recorded_batch.batch_name, 'Recorded batch not found') AS recorded_batch_name,
    coalesce(expected_batch.batch_name, 'Expected batch not found') AS expected_batch_name


FROM rivadataplatform.dataproduct.fact_attendance AS fa
JOIN rivadataplatform.dataproduct.dim_class AS dc
    ON dc.class_key = fa.class_key
LEFT JOIN rivadataplatform.dataproduct.dim_batch AS recorded_batch
    ON recorded_batch.batch_key = fa.batch_key
LEFT JOIN rivadataplatform.dataproduct.dim_batch AS expected_batch
    ON expected_batch.batch_id = dc.batch_id

    
group by dc.class_id, fa.batch_key, dc.batch_id , coalesce(recorded_batch.batch_name, 'Recorded batch not found'), coalesce(expected_batch.batch_name, 'Expected batch not found')
having fa.batch_key IS DISTINCT FROM dc.batch_id
order by dc.class_id;